# Attention Optimizer Results
Loads run histories from local JSONL logs.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

LOGS_DIR = Path('../logs')
SMOOTH_WINDOW = 50

GROUPS = {
    'Baselines': ['BASE-SGD', 'BASE-ADAM', 'BASE-MUON'],
    'History Methods': ['AVG-8', 'AVG-8R', 'ATTNRAW-8', 'ATTNRAW-8R'],
}

LABELS = {
    'BASE-SGD':    'SGD',
    'BASE-ADAM':   'Adam',
    'BASE-MUON':   'Muon',
    'AVG-8':       'Avg-8 (m̃ var)',
    'AVG-8R':      'Avg-8R (raw var)',
    'ATTNRAW-8':   'AttnRaw-8 (m̃ var)',
    'ATTNRAW-8R':  'AttnRaw-8R (raw var)',
}

In [ ]:
def load_metrics(run_id):
    path = LOGS_DIR / run_id / 'metrics.jsonl'
    if not path.exists():
        return None, None
    steps, losses = [], []
    with open(path) as f:
        for line in f:
            d = json.loads(line)
            steps.append(d['step'])
            losses.append(d['loss'])
    return np.array(steps), np.array(losses)

def smooth(x, w):
    if len(x) < w:
        return x
    kernel = np.ones(w) / w
    return np.convolve(x, kernel, mode='valid')

all_runs = {rid: load_metrics(rid) for group in GROUPS.values() for rid in group}
found = [rid for rid, (s, _) in all_runs.items() if s is not None]
print(f"Loaded {len(found)} runs: {found}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.tab20.colors

for i, (run_id, (steps, losses)) in enumerate(all_runs.items()):
    if steps is None:
        continue
    s_loss = smooth(losses, SMOOTH_WINDOW)
    s_steps = steps if len(losses) < SMOOTH_WINDOW else steps[SMOOTH_WINDOW - 1:]
    ax.plot(s_steps, s_loss, label=LABELS.get(run_id, run_id), color=colors[i], lw=1.5)

ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Training Loss — All Runs')
ax.legend(fontsize=8, ncol=3)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes = axes.flatten()

for ax, (group_name, run_ids) in zip(axes, GROUPS.items()):
    for run_id in run_ids:
        steps, losses = all_runs[run_id]
        if steps is None:
            continue
        s_loss = smooth(losses, SMOOTH_WINDOW)
        s_steps = steps if len(losses) < SMOOTH_WINDOW else steps[SMOOTH_WINDOW - 1:]
        ax.plot(s_steps, s_loss, label=LABELS.get(run_id, run_id), lw=1.8)
    ax.set_title(group_name)
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print(f"{'Run ID':<20}  {'Optimizer':<22}  {'Model':<12}  {'Final Loss':>10}")
print('-' * 72)
for run_id, (steps, losses) in all_runs.items():
    if steps is None:
        print(f"{run_id:<20}  {'(not found)':<22}")
        continue
    final = float(np.mean(losses[-min(len(losses), 100):]))
    print(f"{run_id:<20}  {LABELS.get(run_id, run_id):<22}  {'gpt':<12}  {final:>10.4f}")